# Day 3 — Extra exercises: Monte Carlo, Markov chains and Stan

**For two kinds of reader.** If you **missed Day 3** (after informing the lecturer in advance), this notebook is part of your make-up work. If you attended and want **more practice**, work through any part at your own pace; nothing here is required for you.

**What a make-up submission for Day 3 contains** (see `../day-3-makeup.md`):

- **A1–A3 and B1–B3** of this notebook;
- your own solutions to the core in-class exercises in `../day-3-exercises-morning.ipynb` and `../day-3-exercises-afternoon.ipynb`.

The reflection prompts are optional. **C-problems are optional challenges** for everyone. The make-up submission is graded on the 0–4 in-class scale and counts as your in-class score for the missed day.

**Topics.** Part A: Monte Carlo error, and a Metropolis sampler you complete yourself for a posterior that has no formula. Part B: your first Stan model from a template, reading the diagnostics, and a "vague" prior on the wrong scale.

**Background.** Read the Day 3 slides first: `../day-3-slides-morning.pdf` (Monte Carlo, Markov chains, Metropolis, effective sample size) and `../day-3-slides-afternoon.pdf` (Stan, diagnostics). Each task below names the slides it builds on.

**How to work.** Open in JupyterLab, select **R (bayes)**, run the cells in order from a fresh kernel, from this `extra/` folder. Hints are hidden under **Hint**; try first. Write explanations in your own words in the *Your answer* cells: numbers alone are not enough. All data are **synthetic teaching data**; never report them as real measurements.

**Stan.** Part B fits models live. `models/fish-survival.stan` is a **template**: complete the lines marked `TODO` (open the file in JupyterLab's text editor) before fitting. If your installation fails, use UCloud (see the course page) and tell the lecturer early rather than at the deadline.

**Submitting (make-up).** Submit this notebook with outputs, the `.stan` files you edited, and your written answers, together with the in-class exercises, through the announced route by the announced deadline (normally within one week of the missed day). Discussing ideas is fine; the code and explanations must be your own. You may use AI assistants without declaring it, but you should be able to explain every line.

## Setup

Run this cell first.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
library(cmdstanr)    # Stan from R (Part B)
set.seed(2026)
options(width = 100, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")

## Part A — Simulation and a sampler you complete yourself

### A1. Monte Carlo estimates and their error (25 min)

*Background: morning slides "Integrals become averages", "The estimate itself is random", "Where does √S come from?" and "Two kinds of uncertainty".*

**The setting.** In the Day 2 extra exercises, site 1's bird rate λ (birds per hour) had the posterior **Gamma(shape 28, rate 6.5)**. If the rate is λ, the probability of seeing **no birds in a 30-minute watch** is h(λ) = exp(−0.5 λ) (the Poisson probability of 0 with mean 0.5 λ). Averaged over the posterior, the probability is

P(no birds) = E[exp(−0.5 λ) | data],   exact value (6.5 / 7)²⁸.

**Monte Carlo.** Draw S values of λ with `rgamma()`, compute h for each draw, and average. The MCSE of the average is sd(h) / √S.

(a) Estimate P(no birds) with S = 100, 1,000 and 10,000 draws, each with its MCSE. Compare with the exact value.
(b) Estimate P(λ > 5 | data) in the same way (here h is the TRUE/FALSE value `lam > 5`) and compare with `1 - pgamma(5, shape = 28, rate = 6.5)`.
(c) How much smaller does the MCSE get when S is ten times larger? Why does the posterior SD of λ *not* get smaller?

In [ ]:
exact_none <- (6.5 / 7)^28
exact_gt5  <- 1 - pgamma(5, shape = 28, rate = 6.5)

# one Monte Carlo run with S draws
mc_run <- function(S) {
  lam  <- rgamma(S, shape = 28, rate = 6.5)
  none <- exp(-0.5 * lam)       # h(lambda) for (a)
  gt5  <- lam > 5               # h(lambda) for (b)
  tibble(S = S,
         p_none = mean(none), mcse_none = sd(none) / sqrt(S),
         # TODO: p_gt5 and mcse_gt5 in the same way
         sd_lambda = sd(lam))
}

# bind_rows(mc_run(100), mc_run(1000), mc_run(10000))

<details><summary><b>Hint</b> (click to open)</summary>

Each row of the table is one run. The MCSE formula needs only that one run: sd(h) describes how much the h values vary, and dividing by √S gives the spread of their average. Compare the MCSE in the rows S = 1,000 and S = 10,000.

</details>

**Your answer:** *(write here)*

### A2. Complete a Metropolis sampler for a posterior with no formula (30 min)

*Background: morning slides "When we cannot draw directly", "The Metropolis algorithm", "One step, drawn" and "The Metropolis sampler in R"; for the grid, the Day 2 slide "When there is no formula: a grid".*

**The setting.** A fisheries biologist tagged **n = 40 juvenile fish**; **y = 14** were alive after a month. She states her prior not for the survival probability p itself but for its **log-odds**

z = logit(p) = log(p / (1 − p)),   so p = plogis(z).

z can be any real number: z = 0 means p = 0.5, z = −1 means p = 0.27, z = 1 means p = 0.73. Her prior is **z ~ Normal(−1, 0.8)**: survival probably below 50%. This prior is not a beta distribution, so the posterior has **no formula**, and we sample it instead.

- Likelihood: y | z ~ Binomial(40, plogis(z)).
- log(likelihood × prior) = `dbinom(y, n, plogis(z), log = TRUE) + dnorm(z, m0, s0, log = TRUE)`.

**Tasks.**

(a) Complete `log_post(z)` and the accept line in `metropolis()` (marked `TODO`). The sampler is the one from the slides, now for z.
(b) Run 6,000 iterations with `step = 0.6` from z = 0, drop the first 1,000 as warmup, and transform the kept values to p with `plogis()`.
(c) Check your sampler against a **grid** on z (the code is supplied): compare the posterior mean and 90% interval of p. Plot a histogram of your p values.
(d) Why is a check like (c) useful, and why is it not available for models with many parameters?

In [ ]:
n <- 40; y <- 14        # data: 14 of 40 fish survived
m0 <- -1; s0 <- 0.8     # prior: z ~ Normal(-1, 0.8)

log_post <- function(z) {
  # TODO: log likelihood + log prior (see the formula above)
}

metropolis <- function(n_iter, step, start) {
  chain <- numeric(n_iter)
  current <- start
  for (i in 1:n_iter) {
    proposal  <- current + rnorm(1, mean = 0, sd = step)
    log_ratio <- log_post(proposal) - log_post(current)
    # TODO: accept the proposal with probability min(1, exp(log_ratio))
    chain[i] <- current
  }
  chain
}
acceptance_rate <- function(chain) mean(diff(chain) != 0)

# (b)
# set.seed(2026)
# chain_z <- metropolis(6000, step = 0.6, start = 0)
# p_mh <- plogis(chain_z[1001:6000])

# (c) grid check (supplied): z values from -5 to 3, weights = likelihood x prior, normalised
# z_grid <- seq(-5, 3, length.out = 4000)
# weight <- exp(log_post(z_grid)); weight <- weight / sum(weight)
# p_grid <- plogis(z_grid)
# grid mean of p: sum(p_grid * weight)
# grid 5% and 95% quantiles of p: the p values where cumsum(weight) passes 0.05 and 0.95
# p_grid[which(cumsum(weight) >= 0.05)[1]];  p_grid[which(cumsum(weight) >= 0.95)[1]]

<details><summary><b>Hint</b> (click to open)</summary>

`log_post`: `dbinom(y, n, plogis(z), log = TRUE) + dnorm(z, m0, s0, log = TRUE)`. It works for a single z and for a whole vector of z values (needed for the grid). The accept line is the same as on the slide: `if (log(runif(1)) < log_ratio) current <- proposal`. After a rejection nothing changes, and `chain[i] <- current` records the current value again.

</details>

**Your answer:** *(write here)*

### A3. Step size, effective sample size and several chains (25 min)

*Background: morning slides "Effective sample size (ESS)", "Autocorrelation and ESS of our chains", "The step size matters" and "Where to start, and when to start counting"; afternoon slide "R-hat: do the chains agree?".*

(a) Run the sampler with step sizes **0.02, 0.6 and 8** (6,000 iterations from z = 0; keep the last 5,000). For each, report the acceptance rate and the ESS of the kept z values (`posterior::ess_basic()`), and plot the three traces, one panel each.
(b) Run **four chains** with step 0.6 from the starting values −4, −2, 2 and 4. Plot all four traces in one figure (all 6,000 iterations), and compute R-hat of the kept values with `posterior::rhat()`. Repeat with step 0.02.
(c) Why can a high acceptance rate be bad? What does R-hat tell you, and what can it not tell you?

In [ ]:
# (a) three step sizes
# set.seed(2026)
# z_small  <- metropolis(6000, step = 0.02, start = 0)
# z_medium <- metropolis(6000, step = 0.6,  start = 0)
# z_large  <- metropolis(6000, step = 8,    start = 0)
# tibble(step       = c(0.02, 0.6, 8),
#        acceptance = c(acceptance_rate(z_small), ...),
#        ess        = c(posterior::ess_basic(z_small[1001:6000]), ...))
# traces: bind_rows(tibble(iteration = 1:6000, z = z_small, step = "0.02"), ...) |>
#   ggplot(aes(x = iteration, y = z)) + geom_line() + facet_wrap(~ step, ncol = 1)

# (b) four chains from different starting values
# c1 <- metropolis(6000, step = 0.6, start = -4)
# ... c2 (start -2), c3 (start 2), c4 (start 4)
# posterior::rhat(cbind(c1[1001:6000], c2[1001:6000], c3[1001:6000], c4[1001:6000]))

<details><summary><b>Hint</b> (click to open)</summary>

Compare each ESS with the 5,000 values you kept. Look at the traces: does the chain travel across the whole range of z, or does it crawl, or stand still? R-hat compares the variation *between* chains with the variation *within* chains; values well above 1.01 mean the chains disagree.

</details>

**Your answer:** *(write here)*

### Reflection A (optional)

In 80–120 words: explain to a colleague why a Metropolis sampler that accepts 98% of its proposals can still give a poor estimate. Use your ESS numbers.

**Your answer:** *(write here)*

## Part B — The same model in Stan

### B1. Your first Stan model, from a template (30 min)

*Background: afternoon slides "A Stan program has blocks" and "Running it from R".*

Open `models/fish-survival.stan` in JupyterLab's text editor. The `data` and `parameters` blocks are written; three lines are marked `TODO`.

(a) Complete the template: the prior on z, the likelihood, and in `generated quantities` a simulated number of survivors `y_new` among 40 new fish. Save the file.
(b) Compile and fit it with the A2 data (4 chains, `seed = 2026`).
(c) Compare the posterior mean and 90% interval of p with your Metropolis and grid results from A2.
(d) What are the posterior mean and 90% interval of `y_new`? Why is the interval wider than 40 × (the interval of p)?

In [ ]:
fish_data <- list(n = 40, y = 14, m0 = -1, s0 = 0.8)

# mod_fish <- cmdstan_model("models/fish-survival.stan")
# fit_fish <- mod_fish$sample(data = fish_data, chains = 4, seed = 2026, refresh = 0)
# fit_fish$summary(c("z", "p", "y_new"))
# draws_fish <- fit_fish$draws(format = "df")

<details><summary><b>Hint</b> (click to open)</summary>

In the model block: `z ~ normal(m0, s0);` and `y ~ binomial(n, inv_logit(z));` (`inv_logit` is Stan's name for `plogis`; `y ~ binomial_logit(n, z);` does the same and is more stable). In generated quantities: `int y_new = binomial_rng(n, p);`. If compilation fails, read the **first** error message: Stan names the line and column.

</details>

**Your answer:** *(write here)*

### B2. Diagnostics before interpretation (20 min)

*Background: afternoon slides "Six numbers to check after every fit", "R-hat: do the chains agree?", "Reading a real Stan summary" and "Two different checks".*

(a) For the B1 fit, report R-hat, bulk and tail ESS, divergences, max-treedepth hits and E-BFMI (`fit_fish$summary()` and `fit_fish$diagnostic_summary()`). Do they support reporting the posterior?
(b) Now deliberately run the same model with **far too few iterations**: add `iter_warmup = 15, iter_sampling = 30` to `$sample()`. Report the same diagnostics and the posterior mean of p.
(c) Name one question about the *model* (not the computation) that none of these diagnostics can answer.

In [ ]:
# (a) fit_fish$summary("p", "mean", "rhat", "ess_bulk", "ess_tail");  fit_fish$diagnostic_summary()
# (b) fit_short <- mod_fish$sample(data = fish_data, chains = 4, seed = 2026, refresh = 0,
#                                  iter_warmup = 15, iter_sampling = 30)

<details><summary><b>Hint</b> (click to open)</summary>

With 15 warmup iterations Stan has hardly tuned itself. Compare R-hat with 1.01 and the ESS with 400. The posterior mean may still *look* reasonable: that is exactly why we check the diagnostics instead of judging whether the answer seems plausible.

</details>

**Your answer:** *(write here)*

### B3. A "vague" prior on the wrong scale (25 min)

*Background: afternoon slides "Running it from R" (the prior's numbers are data: no recompiling) and "Two different checks"; the Day 2 idea of simulating what a prior expects.*

A colleague suggests z ~ Normal(0, **10**) as an "uninformative" prior.

(a) Simulate 10,000 values of p = plogis(z) under this prior and under Normal(−1, 0.8), and plot both histograms of p. What does the "vague" prior actually believe about survival?
(b) Fit the Stan model with m0 = 0, s0 = 10 (only the data list changes) and compare the posterior of p with B1.
(c) Suppose instead only **5** fish had been tagged and **0** survived. Fit both priors to these data and compare the posterior medians and 90% intervals of p. When does the choice of prior matter?

In [ ]:
# (a) what each prior believes about p
# prior_p <- bind_rows(
#   tibble(p = plogis(rnorm(10000, 0, 10)),  prior = "z ~ Normal(0, 10)"),
#   tibble(p = plogis(rnorm(10000, -1, 0.8)), prior = "z ~ Normal(-1, 0.8)"))
# ggplot(prior_p, aes(x = p)) + geom_histogram(binwidth = 0.02) + facet_wrap(~ prior, ncol = 1)

# (b) fit_vague <- mod_fish$sample(data = list(n = 40, y = 14, m0 = 0, s0 = 10), chains = 4, seed = 2026, refresh = 0)
# (c) small data: list(n = 5, y = 0, m0 = ..., s0 = ...)

<details><summary><b>Hint</b> (click to open)</summary>

plogis(10) is essentially 1 and plogis(−10) essentially 0. A Normal(0, 10) prior on the log-odds puts most of its mass far out in the tails. With 40 fish the data dominate; with 5 fish and 0 survivors they do not.

</details>

**Your answer:** *(write here)*

### Reflection B (optional)

In 80–120 words: give one **computation check** and one **model check** for the fish-survival analysis, and explain why passing the first does not mean passing the second.

**Your answer:** *(write here)*

## C. Optional challenges

**C1. A stationary distribution** (*morning slides "A Markov chain: tomorrow depends on today", "The stationary distribution" and "One long chain settles at π"*). A shared bike ends each day at one of three stations, A, B or C. Where it ends tomorrow depends only on where it ended today:

| today ↓ tomorrow → | A | B | C |
|---|---|---|---|
| A | 0.6 | 0.3 | 0.1 |
| B | 0.2 | 0.5 | 0.3 |
| C | 0.1 | 0.3 | 0.6 |

(a) The bike starts at A. Compute the probabilities of A, B and C on days 1 to 30 by repeating `prob <- prob %*% P`, and plot them. (b) Do the same starting from C. (c) Simulate one chain of 10,000 days with `sample()` and compare the fraction of days at each station with (a). (d) Check that the long-run probabilities π are stationary: `pi %*% P` gives π again.

In [ ]:
P <- matrix(c(0.6, 0.3, 0.1,
              0.2, 0.5, 0.3,
              0.1, 0.3, 0.6), nrow = 3, byrow = TRUE)

# (a) prob <- c(1, 0, 0)            # day 0: at A
#     for (day in 1:30) prob <- prob %*% P
# (c) station <- numeric(10000); current <- 1
#     for (t in 1:10000) { current <- sample(1:3, 1, prob = P[current, ]); station[t] <- current }
#     table(station) / 10000

**C2. The gamma–Poisson model in Stan** (*Day 2 afternoon slide "The same model in Stan"*). Complete `models/poisson-exposure.stan` (only a skeleton is given) for the Day 2 bird counts at site 1 (counts 3, 7, 2, 5, 9 in 0.5, 1.5, 0.5, 1 and 2 hours) with a Gamma(2, 1) prior. Check that Stan reproduces the exact Gamma(28, 6.5) posterior: mean 28 / 6.5 and 90% interval `qgamma(c(0.05, 0.95), shape = 28, rate = 6.5)`.

In [ ]:
bird_data <- list(N = 5, count = c(3, 7, 2, 5, 9), hours = c(0.5, 1.5, 0.5, 1, 2), a0 = 2, b0 = 1)
# mod_bird <- cmdstan_model("models/poisson-exposure.stan")

**Your answer:** *(write here)*

## Make-up checklist

- [ ] A1–A3 and B1–B3 · [ ] in-class exercises (`../day-3-exercises-morning.ipynb` and `../day-3-exercises-afternoon.ipynb`)
- [ ] Your completed `models/fish-survival.stan` is submitted with the notebook
- [ ] Diagnostics are reported with numbers before any posterior is interpreted
- [ ] The notebook runs from a fresh kernel